In [6]:

import os
import getpass

from langchain.agents import Tool, AgentType, initialize_agent
from langchain.memory import ConversationBufferMemory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    HarmBlockThreshold,
    HarmCategory,
)
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.utilities import DuckDuckGoSearchAPIWrapper
from langchain.agents import AgentExecutor
from langchain import hub, LLMMathChain
from langchain.agents.format_scratchpad import format_log_to_str
from langchain.agents.output_parsers import ReActSingleInputOutputParser
from langchain.tools.render import render_text_description

In [4]:
#@title Setting up the Auth
import os
import google.generativeai as genai
from dotenv import load_dotenv, dotenv_values
load_dotenv()
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
genai.configure(api_key=GOOGLE_API_KEY)

/home/gigadelux/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# setup model
llm = ChatGoogleGenerativeAI(
    model="gemini-1.0-pro",
    convert_system_message_to_human=True,
    handle_parsing_errors=True,
    temperature=0.6,
    max_tokens= 200,
    safety_settings = {
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
    },
)

In [20]:
models

[Model(name='models/chat-bison-001',
       base_model_id='',
       version='001',
       display_name='PaLM 2 Chat (Legacy)',
       description='A legacy text-only model optimized for chat conversations',
       input_token_limit=4096,
       output_token_limit=1024,
       supported_generation_methods=['generateMessage', 'countMessageTokens'],
       temperature=0.25,
       top_p=0.95,
       top_k=40),
 Model(name='models/text-bison-001',
       base_model_id='',
       version='001',
       display_name='PaLM 2 (Legacy)',
       description='A legacy model that understands text and generates text as an output',
       input_token_limit=8196,
       output_token_limit=1024,
       supported_generation_methods=['generateText', 'countTextTokens', 'createTunedTextModel'],
       temperature=0.7,
       top_p=0.95,
       top_k=40),
 Model(name='models/embedding-gecko-001',
       base_model_id='',
       version='001',
       display_name='Embedding Gecko',
       description='Obtai

In [23]:

embedder = genai.get_model("models/text-embedding-004")
vector = genai.embed_content(
    model=embedder,
    content="I want to search people to love", 
)
vector

{'embedding': [-0.024785874,
  -0.0058527756,
  -0.016679026,
  -0.008591824,
  0.016187081,
  0.044620357,
  0.062054344,
  0.01920355,
  -0.003781143,
  0.01036664,
  -0.030985365,
  0.014551959,
  0.03417492,
  -0.0014090944,
  -0.035385318,
  -0.088832974,
  -0.018275792,
  0.04517761,
  -0.054403927,
  -0.023073658,
  0.019893875,
  0.016760852,
  -0.02585453,
  -0.026101261,
  -0.020169519,
  0.013892892,
  -0.04101895,
  -0.008548068,
  0.0507871,
  -0.00095079147,
  0.01012917,
  0.054844894,
  0.04867114,
  0.02346799,
  -0.02052504,
  0.010399808,
  -0.020298718,
  -0.045977954,
  0.042065077,
  -0.051136587,
  -0.004129798,
  0.041848775,
  -0.032845687,
  -0.0035809306,
  -0.00267762,
  -0.027925745,
  0.013426789,
  0.042759463,
  -0.043551497,
  0.06371411,
  0.036444888,
  -0.034118667,
  0.047628216,
  0.024524538,
  -0.03587018,
  -0.02778455,
  -0.024522858,
  -0.05544323,
  0.018768262,
  -0.03383588,
  0.029053213,
  -0.018396428,
  -0.041076124,
  -0.024690542,
  0

In [24]:
len(vector["embedding"])

768

In [ ]:
tools = [
    Tool(
        name="DuckDuckGo Search",
        func=ddg_search.run,
        description="Useful to browse information from the internet to know recent results and information you don't know. Then, tell user the result.",
    ),
    Tool.from_function(
        func=llm_math_chain.run,
        name="Calculator",
        description="Useful for when you need to answer questions about very simple math. This tool is only for math questions and nothing else. Only input math expressions. Not supporting symbolic math.",
    ),
]

In [ ]:
# To query Gemini
llm_prompt_template = """Write a concise summary of the following:
"{text}"
CONCISE SUMMARY:"""
llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

In [ ]:
memory = ConversationBufferMemory(memory_key="chat_history")

In [ ]:
agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_log_to_str(x["intermediate_steps"]),
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm_with_stop
    | ReActSingleInputOutputParser()
)

In [ ]:

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, memory=memory)

In [2]:
system_template = """You are an accurate ai assistant that moderate and filtering any kind of request you receive as input. 
    Those requests are from an app that help you find the people similar to the request through ai. 
    Your input will be a request about finding those people or a json of the people plus the request in form of 
    "request":request, "people": [{"name":name, "jobtitle": jobtitle, "description":description}, ...]. 
    Your output will always be in json with 3 key/values pairs: "response": your response message, 
    "validRequest": a boolean that serves as moderation for the input and will be true 
    ONLY if is not a filtering operation and not with the welcome message, 
    "optimizedPrompt": optimize the request with a single string with all the topics prompted in an extremely accurate way, 
    if the request is empty output a welcome message"""

In [8]:
llm.generate(messages=[SystemMessage(content=system_template),HumanMessage(content="find me people to talk about one piece")])

/home/gigadelux/.local/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:344: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


ValueError: Unexpected message with type <class 'tuple'> at the position 0.